In [1]:
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from chiricoca.geo.utils import to_point_geodataframe
from cytoolz import valfilter, keymap
from gensim.utils import deaccent
from geopy.geocoders import Nominatim


In [ ]:
elecciones = pd.read_csv(
    "data/presidenciales_2021/Servel_20211121_PRESIDENCIALES_CHILE.csv",
    sep=";",
)
elecciones.head()

In [ ]:
elecciones["location_name"] = (
    elecciones["local_nombre"].str.replace(r"LOCAL\s?:\s?\d+", "").str.strip()
)
elecciones["location_name"]


In [5]:
elecciones["region_nombre"] = elecciones["region_nombre"].str.strip()


In [6]:
elecciones_region = elecciones[
    elecciones["region_nombre"] == "METROPOLITANA DE SANTIAGO"
].copy()


In [ ]:
servel_locations = (
    pd.read_excel("data/servel/Locales_de_votacion.xlsx")
    .pipe(lambda x: x[x["Región"] == "METROPOLITANA DE SANTIAGO"])
    .assign(
        local=lambda x: x["Local de Votación"]
        .str.replace(r"LOCAL\s?:\s?\d+", "")
        .str.strip()
    )
    .drop_duplicates(subset=["Local de Votación"])
)
servel_locations


In [ ]:
clean_name = lambda x: (
    x.replace(" PARTICULAR", "")
    .replace("NRO ", "N°")
    .replace("LOCAL: 1", "")
    .replace("LOCAL :1", "")
    .replace("LOCAL: 2", "")
    .replace("LOCAL :2", "")
    .replace("LOCAL: 3", "")
    .replace("LOCAL :3", "")
    .replace("LOCAL: 4", "")
    .replace("LOCAL :4", "")
    .replace("LOCAL: 5", "")
    .replace("LOCAL: 6", "")
    .replace("LOCAL: 7", "")
    .replace("LOCAL: 8", "")
    .replace("LOCAL: 9", "")
    .replace("ESC.", "ESCUELA")
    .replace("ESC ", "ESCUELA ")
    .replace("EDUC.", "EDUCACIONAL")
    .replace("EDUC ", "EDUCACIONAL ")
    .replace("POLIV ", "POLIVALENTE ")
    .replace("POLIV.", "POLIVALENTE")
    .replace("U.", "UNIVERSIDAD")
    .strip()
)

servel_locations["clean_name"] = servel_locations["local"].map(clean_name)
elecciones_region['clean_name'] = elecciones_region['location_name'].map(clean_name)

servel_locations

In [ ]:
locations = pd.Series(elecciones_region["clean_name"].unique())
locations.shape


In [ ]:
location_addresses = (
    servel_locations.drop_duplicates(subset="clean_name")
    .set_index("clean_name")
    .apply(
        lambda x: x["Dirección"]
        + ", "
        + x["Comuna"]
        + ", Región Metropolitana de Santiago, Chile",
        axis=1,
    )
    .to_dict()
)
len(location_addresses), location_addresses

In [84]:
geolocator = Nominatim(user_agent="chilean_voting_location_analysis")

In [85]:
location_coords = {}


In [ ]:
errors = 0

for loc in locations.values:
    if loc in location_coords:
        continue
    
    try:
        result = geolocator.geocode(
            f"{loc}, Región Metropolitana de Santiago, Chile"
        )

        if result is None and loc in location_addresses:
            time.sleep(1)
            result = geolocator.geocode(location_addresses[loc])

        location_coords[loc] = result
        print(loc, location_coords[loc])
    except Exception as e:
        print("error", e)
        errors += 1
        if errors >= 3:
            break
    finally:
        time.sleep(1)

location_coords


In [ ]:
len(valfilter(lambda x: x is not None, location_coords)), len(
    valfilter(lambda x: x is None, location_coords)
)


In [ ]:
from chiricoca.geo.utils import clip_point_geodataframe

scl_bounds = [-70.88006218, -33.67612715, -70.43015094, -33.31069169]

# scl_zones = clip_area_geodataframe(zones.to_crs('epsg:4326'), scl_bounds).to_crs(zones.crs)

location_points = pd.DataFrame(
    [
        {"location": x[0], "x": x[1].longitude, "y": x[1].latitude}
        for x in valfilter(lambda x: x, location_coords).items()
    ]
).pipe(
    lambda x: clip_point_geodataframe(
        to_point_geodataframe(x, "x", "y", drop=True), scl_bounds
    )
)
location_points.plot()

In [119]:
import h3
import geopandas as gpd
from shapely.geometry import Polygon

def h3_grid(bounds, extra_margin=0.0, grid_level=12, crs="epsg:4326"):
    class MockGeo:
        def __init__(self, d):
            self.d = d

        @property
        def __geo_interface__(self):
            return self.d
    
    bounds = list(bounds)
    bounds[0] = bounds[0] - extra_margin * (bounds[2] - bounds[0])
    bounds[2] = bounds[2] + extra_margin * (bounds[2] - bounds[0])
    bounds[1] = bounds[1] - extra_margin * (bounds[3] - bounds[1])
    bounds[3] = bounds[3] + extra_margin * (bounds[3] - bounds[1])

    cell_ids = h3.h3shape_to_cells(
        h3.geo_to_h3shape(
            MockGeo(
                {
                    "type": "Polygon",
                    "coordinates": [
                        [
                            [bounds[0], bounds[1]],
                            [bounds[2], bounds[1]],
                            [bounds[2], bounds[3]],
                            [bounds[0], bounds[3]],
                        ]
                    ],
                }
            )
        ),
        res=grid_level,
    )

    h3grid = gpd.GeoDataFrame(
        {"h3_cell_id": list(map(str, cell_ids))},
        geometry=[
            Polygon(
                list(
                    map(lambda x: tuple(reversed(x)), h3.cell_to_boundary(cell_id))
                )
            )
            for cell_id in cell_ids
        ],
        crs="epsg:4326",
    ).to_crs(crs)

    return h3grid

In [ ]:
grid = h3_grid(location_points.total_bounds, extra_margin=0.05, grid_level=8)
ax = grid.plot(facecolor='none', edgecolor='#abacab')
location_points.plot(ax=ax, color='magenta', markersize=1)

In [ ]:
locations_in_grid = gpd.sjoin(location_points, grid, op="within", how="inner")
locations_in_grid.plot()


In [128]:
grid.to_parquet('results/07_grid.parquet')

In [129]:
locations_in_grid.to_parquet('results/07_locations_in_grid.parquet')